In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer #embedding model will be here
import chromadb
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.vectorstores import FAISS
import pickle
from langchain.embeddings.base import Embeddings


c:\Year 4\GradProject\YAML-Wizard\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class EmbeddingManager(Embeddings):
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"): #model that converts text into vectors
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model is loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading the model: {e}")
            raise

    def embed_documents(self, texts)-> np.ndarray:
        if not self.model:
            raise ValueError("No Model") 
        print(f"Generating Embeddings for {len(texts)}")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    def embed_query(self, query:str)-> np.ndarray:
        if not self.model:
            raise ValueError("No Model") 
        print(f"Generating Embeddings for {query}")
        embeddings = self.model.encode(query, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2
Model is loaded successfully. Embedding dimension: 384


In [23]:
def save_to_faiss(document_chunks, vectorstore_filename="faiss_vectorstore.pkl"):
    faiss_db = FAISS.from_documents(document_chunks, embedding_manager)

    with open(vectorstore_filename, "wb") as f:
        pickle.dump(faiss_db, f)

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
loader = CSVLoader("../data/GitHubActions_Docs_modified.csv",encoding='utf-8')
data = loader.load()
#data

In [ ]:
save_to_faiss(data)

In [14]:
texts_2=[text.page_content for text in data]

In [5]:
faiss_db = FAISS.from_documents(data,embedding=embedding_manager)

Generating Embeddings for 1318


Batches: 100%|██████████| 42/42 [00:17<00:00,  2.40it/s]

Generated embeddings with shape: (1318, 384)


In [6]:
faiss_db.similarity_search("run job directly on runner machine",k=2)

Generating Embeddings for run job directly on runner machine


Batches: 100%|██████████| 1/1 [00:00<00:00, 55.56it/s]

Generated embeddings with shape: (384,)


[Document(id='e7be6c31-19e1-48b7-96bc-675a14938f9a', metadata={'source': '../data/GitHubActions_Docs_modified.csv', 'row': 167}, page_content='Title: Choosing the runner for a job - Overview\nDescription: Use jobs.<job_id>.runs-on to define the type of machine to run the job on. You can target runners based on the labels assigned to them, or their group membership, or a combination of these. You can provide runs-on as: If you specify an array of strings or variables, your workflow will execute on any runner that matches all of the specified runs-on values. For example, here the job will only run on a self-hosted runner that has the labels linux , x64 , and gpu :\nCode: runs-on: [self-hosted, linux, x64, gpu]'),
 Document(id='3dd075a1-0d75-4e0e-b0d8-738539d31ec6', metadata={'source': '../data/GitHubActions_Docs_modified.csv', 'row': 117}, page_content="Title: Creating PostgreSQL service containers - Running jobs directly on the runner machine\nDescription: When you run a job directly on

In [28]:
results = faiss_db.similarity_search_with_score("Creating PostgreSQL service containers",k=2)
results


Generating Embeddings for Creating PostgreSQL service containers


Batches: 100%|██████████| 1/1 [00:00<00:00, 100.05it/s]

Generated embeddings with shape: (384,)


[(Document(id='a1eae146-376f-4d47-a0b1-86e52f2b295d', metadata={'source': '../data/GitHubActions_Docs_modified.csv', 'row': 114}, page_content="Title: Creating PostgreSQL service containers - Running jobs in containers\nDescription: Configuring jobs to run in a container simplifies networking configurations between the job and the service containers. Docker containers on the same user-defined bridge network expose all ports to each other, so you don't need to map any of the service container ports to the Docker host. You can access the service container from the job container using the label you configure in the workflow. You can copy this workflow file to the .github/workflows directory of your repository and modify it as needed.\nCode: name: PostgreSQL service example\non: push\n\njobs:\n  # Label of the container job\n  container-job:\n    # Containers must run in Linux based operating systems\n    runs-on: ubuntu-latest\n    # Docker Hub image that `container-job` executes in\n    

In [ ]:
class RAGRetriever:
    def __init__(self, database, embedding_manager:EmbeddingManager):
        self.vector_store = database
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int = 2, score_threshold: float= 0.0) -> List[Dict[str,Any]]:
        try:
            results = faiss_db.similarity_search_with_score(query,k=2)
            print(f"Debug: asked for {top_k}, got {len(results)}")
            retrieved_docs=[]

            if results:
                documents=[]
                metadatas=[]
                ids=[]
                distances=[]
                for result,score in results:
                    documents.append(result.page_content)
                    print("here")
                    metadatas.append(result.metadata)
                    distances.append(score)
                    ids.append(result.id)

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - (distance / 2.0)
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
            
            print(f"Debug: returning {len(retrieved_docs)} results")
            return retrieved_docs
            
        except Exception as e:
            print(f"Error retrieval: {e}")
            return []

rag_retriever=RAGRetriever(faiss_db,embedding_manager)
rag_retriever
results=rag_retriever.retrieve("run job directly on runner machine")
results

Generating Embeddings for Creating PostgreSQL service containers


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.83it/s]

Generated embeddings with shape: (384,)
Debug: asked for 2, got 2
here
here
Debug: returning 2 results


[{'id': 'a1eae146-376f-4d47-a0b1-86e52f2b295d',
  'content': "Title: Creating PostgreSQL service containers - Running jobs in containers\nDescription: Configuring jobs to run in a container simplifies networking configurations between the job and the service containers. Docker containers on the same user-defined bridge network expose all ports to each other, so you don't need to map any of the service container ports to the Docker host. You can access the service container from the job container using the label you configure in the workflow. You can copy this workflow file to the .github/workflows directory of your repository and modify it as needed.\nCode: name: PostgreSQL service example\non: push\n\njobs:\n  # Label of the container job\n  container-job:\n    # Containers must run in Linux based operating systems\n    runs-on: ubuntu-latest\n    # Docker Hub image that `container-job` executes in\n    container: node:20-bookworm-slim\n\n    # Service containers to run with `containe